In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
PATH_AVT = '/resorces/Neftcode_2.0/data/avt_tags.csv'
PATH_HDT = '/resorces/Neftcode_2.0/data/242000_tags.csv'
PATH_PAK = '/resorces/Neftcode_2.0/Выгрузка ПАК 01.01.2023 - н.в_.xlsx'
PATH_LIMS = '/resorces/Neftcode_2.0/ЛИМСы 01.01.2023 - н.в_ (2).xlsx'

In [ ]:
# === Умный парсер дат ===
def parse_russian_dates(date_series):
    # 1. Пытаемся распарсить всё как есть (если Excel уже сам перевел часть ячеек в даты)
    parsed = pd.to_datetime(date_series, errors='coerce')

    # 2. Ищем те ячейки, которые выдали ошибку (скорее всего это текст типа "17-окт-25 08:00:00")
    mask = parsed.isna() & date_series.notna()

    if mask.any():
        s = date_series[mask].astype(str).str.lower().str.strip()
        ru_months = {'янв': '01', 'фев': '02', 'мар': '03', 'апр': '04', 'май': '05', 'июн': '06',
                     'июл': '07', 'авг': '08', 'сен': '09', 'окт': '10', 'ноя': '11', 'дек': '12'}
        for ru, num in ru_months.items():
            s = s.str.replace(ru, num, regex=False)

        # Парсим очищенный текст, разрешая pandas самому понять формат (указываем dayfirst=True)
        parsed_text = pd.to_datetime(s, errors='coerce', dayfirst=True)
        # Подставляем спасенные даты обратно
        parsed.loc[mask] = parsed_text

    return parsed

# === Умный парсер чисел ===
def clean_numeric(val_series):
    # Меняем запятые на точки, убираем случайные пробелы тысячных разрядов
    s = val_series.astype(str).str.replace(',', '.', regex=False).str.replace(' ', '', regex=False)
    # to_numeric превратит любой мусор (типа 'Pt Created') в пустоту (NaN)
    return pd.to_numeric(s, errors='coerce')


In [ ]:

# Загружаем, парсим даты, делаем сортировку (ОБЯЗАТЕЛЬНО для слияний)
avt = pd.read_csv(PATH_AVT, parse_dates=['date']).sort_values('date')
hdt = pd.read_csv(PATH_HDT, parse_dates=['date']).sort_values('date')

# Т.к. шаг одинаковый, можно слить через merge_asof (надежнее) или merge(how='outer')
df_telemetry = pd.merge_asof(avt, hdt, on='date', direction='nearest', tolerance=pd.Timedelta('2min'))
print("1. Обработали телеметрию...")

1. Обработали телеметрию...


In [ ]:

df_pak_raw = pd.read_excel(PATH_PAK, header=None)
# Вручную вытаскиваем два нужных куска таблицы
# Кусок 1: Сера (Столбцы 0 и 1)
name_sulfur = df_pak_raw.iloc[0, 0] # Достаем название из ячейки A1
pak_sulfur = df_pak_raw.iloc[2:, [0, 1]].copy() # Берем данные начиная с 3-й строки
pak_sulfur.columns = ['date', name_sulfur]
pak_sulfur['date'] = parse_russian_dates(pak_sulfur['date'])
pak_sulfur[name_sulfur] = clean_numeric(pak_sulfur[name_sulfur])
pak_sulfur = pak_sulfur.dropna(subset=['date', name_sulfur]).sort_values('date')

# Кусок 2: D15 (Столбцы 3 и 4)
name_d15 = df_pak_raw.iloc[0, 3] # Достаем название из ячейки D1
pak_d15 = df_pak_raw.iloc[2:, [3, 4]].copy()
pak_d15.columns = ['date', name_d15]
pak_d15['date'] = parse_russian_dates(pak_d15['date'])
pak_d15[name_d15] = clean_numeric(pak_d15[name_d15])
pak_d15 = pak_d15.dropna(subset=['date', name_d15]).sort_values('date')

# Сливаем оба анализатора ПАК в одну таблицу (outer join, т.к. даты у них разные)
df_pak_clean = pd.merge(pak_sulfur, pak_d15, on='date', how='outer').sort_values('date')

# Подтягиваем ПАК к телеметрии
df_main = pd.merge_asof(df_telemetry, df_pak_clean, on='date', direction='backward', tolerance=pd.Timedelta('1 hour'))
print("2. Обработали ПАК...")

2. Обработали ПАК...


In [ ]:
print("3. Обработка ЛИМС...")
df_lims_raw = pd.read_excel(PATH_LIMS, header=None)

# Так как в 1 строке объединенные ячейки, "протягиваем" названия установок вправо
installations = df_lims_raw.iloc[0].ffill()

lims_dfs = [] # Сюда будем складывать датафреймы по каждому показателю
# Идем по таблице парами столбцов: (0,1), (2,3), (4,5) ...
for i in range(0, df_lims_raw.shape[1], 2):
    val_param = df_lims_raw.iloc[1, i]
    # Если во второй строке пусто, пропускаем
    if pd.isna(val_param) or str(val_param).strip() in ['', 'nan']:
        continue

    # Собираем красивое уникальное имя: "Установка - Параметр"
    inst_name = str(installations[i]).replace("Установка '", "").replace("'", "")
    param_name = str(val_param)
    full_col_name = f"LIMS_{inst_name}_{param_name}"

    # Вытаскиваем даты и значения (с 4-й строки вниз)
    part = df_lims_raw.iloc[4:, [i, i+1]].copy()
    part.columns = ['date', full_col_name]

    # Очищаем данные
    part['date'] = parse_russian_dates(part['date'])
    part[full_col_name] = clean_numeric(part[full_col_name]) # Pt Created тут превратится в NaN
    part = part.dropna(subset=['date', full_col_name]).sort_values('date')

    if not part.empty:
        lims_dfs.append(part)
        print(f"Успешно спарсили: {full_col_name} (записей: {len(part)})")

if len(lims_dfs) == 0:
    raise ValueError("Скрипт не смог распознать ни одной колонки из файла ЛИМС. Убедитесь, что формат файла не изменился.")

# Сливаем все лабораторные тесты в одну большую таблицу ЛИМС
df_lims_clean = lims_dfs[0]
for part in lims_dfs[1:]:
    df_lims_clean = pd.merge(df_lims_clean, part, on='date', how='outer')
df_lims_clean = df_lims_clean.sort_values('date')

# Создаем колонку с временем взятия пробы ЛИМС
df_lims_clean['lims_timestamp'] = df_lims_clean['date']

# Финальный джоин: подтягиваем ЛИМС к нашему главному датасету
df_final = pd.merge_asof(df_main, df_lims_clean, on='date', direction='backward')

# Считаем возраст анализа в часах
df_final['lims_age_hours'] = (df_final['date'] - df_final['lims_timestamp']).dt.total_seconds() / 3600


print("\nГОТОВО! Итоговый датасет собран.")
print(f"Размер итоговой таблицы: {df_final.shape}")
print("Фрагмент готовых данных:")

3. Обработка ЛИМС...
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CFPP (записей: 44)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_90%.T (записей: 580)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_50%.T (записей: 648)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_EBP.T (записей: 648)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_PourPoint (записей: 1255)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_D15 (записей: 219)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_I350 (записей: 138)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CloudPoint (записей: 13)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CloudPoint_1 (записей: 112)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_95%.T (записей: 649)
Успешно спарсили: LIMS_АВТ. Точка отбора 1. Продукт Дизель

In [ ]:

display(df_final[['date', '24-2000:Mg.Sulfur', '24-2000:D15', 'LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CFPP', 'lims_age_hours']].tail(10))

,date,24-2000:Mg.Sulfur,24-2000:D15,LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CFPP,lims_age_hours
189207,2026-08-06 22:30:00,7.760094,835.3033,NaN,8.500000
189208,2026-08-06 22:40:00,7.939949,835.2109,NaN,8.666667
189209,2026-08-06 22:50:00,8.019502,835.0686,NaN,8.833333
189210,2026-08-06 23:00:00,8.021791,835.4318,NaN,9.000000
189211,2026-08-06 23:10:00,8.033235,835.0873,NaN,9.166667
189212,2026-08-06 23:20:00,8.007294,835.1230,NaN,9.333333
189213,2026-08-06 23:30:00,7.706686,835.4171,NaN,9.500000
189214,2026-08-06 23:40:00,7.392977,835.5061,NaN,9.666667
189215,2026-08-06 23:50:00,7.394635,835.6401,NaN,9.833333
189216,2026-08-07 00:00:00,7.904419,835.6633,NaN,10.000000


In [ ]:

display(df_final[['date', '24-2000:Mg.Sulfur', '24-2000:D15', 'LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CFPP', 'lims_age_hours']].head(10))

,date,24-2000:Mg.Sulfur,24-2000:D15,LIMS_АВТ. Точка отбора 1. Продукт Дизельное топливо_CFPP,lims_age_hours
0,2023-01-01 00:00:00,6.572924,NaN,NaN,NaN
1,2023-01-01 00:10:00,6.730094,NaN,NaN,NaN
2,2023-01-01 00:20:00,7.158879,NaN,NaN,NaN
3,2023-01-01 00:30:00,7.620472,NaN,NaN,NaN
4,2023-01-01 00:40:00,8.173620,NaN,NaN,NaN
5,2023-01-01 00:50:00,8.716105,NaN,NaN,NaN
6,2023-01-01 01:00:00,9.642323,NaN,NaN,NaN
7,2023-01-01 01:10:00,9.722434,NaN,NaN,NaN
8,2023-01-01 01:20:00,9.378595,NaN,NaN,NaN
9,2023-01-01 01:30:00,9.051027,NaN,NaN,NaN
